In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.metrics         import (accuracy_score, f1_score, precision_score,
                                     recall_score, roc_auc_score,
                                     average_precision_score, confusion_matrix,
                                     precision_recall_curve, roc_curve)
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings("ignore")
os.makedirs("outputs", exist_ok=True)
print("All libraries loaded!")

In [ ]:
data = pd.read_csv("smart_grid_stability_augmented.csv")

data = data.rename(columns={
    "tau1" : "reaction_producer",
    "tau2" : "reaction_consumer1",
    "tau3" : "reaction_consumer2",
    "tau4" : "reaction_consumer3",
    "p1"   : "power_producer",
    "p2"   : "power_consumer1",
    "p3"   : "power_consumer2",
    "p4"   : "power_consumer3",
    "g1"   : "price_sensitivity_producer",
    "g2"   : "price_sensitivity_consumer1",
    "g3"   : "price_sensitivity_consumer2",
    "g4"   : "price_sensitivity_consumer3",
    "stab" : "stability_score",
    "stabf": "grid_status",
})

print(f"Rows    : {len(data):,}")
print(f"Columns : {list(data.columns)}")
data.head()

In [ ]:
print("Missing values  :", data.isnull().sum().sum(),  "← should be 0")
print("Duplicate rows  :", data.duplicated().sum(),    "← should be 0")
print()
print("Label counts:")
print(data["grid_status"].value_counts())

In [ ]:
plt.figure(figsize=(16, 10))
plt.suptitle("EDA — Understanding the Grid Stability Data", fontsize=14)

# ── Chart 1: Stable vs Unstable count ───────────────────
plt.subplot(2, 3, 1)
counts = data["grid_status"].value_counts()
plt.bar(counts.index, counts.values, color=["steelblue", "tomato"])
plt.title("Stable vs Unstable Count")
plt.ylabel("Number of rows")
for i, val in enumerate(counts.values):
    plt.text(i, val + 200, f"{val:,}", ha="center", fontweight="bold")

In [ ]:
# ── Chart 2: Stability score by class ───────────────────
# Negative score = unstable, positive = stable
plt.subplot(2, 3, 2)
sns.histplot(data[data["grid_status"]=="stable"]["stability_score"],
             bins=50, color="steelblue", alpha=0.6, label="Stable")
sns.histplot(data[data["grid_status"]=="unstable"]["stability_score"],
             bins=50, color="tomato", alpha=0.6, label="Unstable")
plt.axvline(0, color="black", linestyle="--", label="Boundary = 0")
plt.title("Stability Score by Class")
plt.xlabel("stability_score")
plt.legend()

# ── Chart 3: Reaction producer by class ─────────────────
plt.subplot(2, 3, 3)
sns.histplot(data[data["grid_status"]=="stable"]["reaction_producer"],
             bins=40, color="steelblue", alpha=0.6, label="Stable")
sns.histplot(data[data["grid_status"]=="unstable"]["reaction_producer"],
             bins=40, color="tomato", alpha=0.6, label="Unstable")
plt.title("Reaction Producer by Class")
plt.xlabel("reaction_producer")
plt.legend()

# ── Chart 4: Power producer by class ────────────────────
plt.subplot(2, 3, 4)
sns.histplot(data[data["grid_status"]=="stable"]["power_producer"],
             bins=40, color="steelblue", alpha=0.6, label="Stable")
sns.histplot(data[data["grid_status"]=="unstable"]["power_producer"],
             bins=40, color="tomato", alpha=0.6, label="Unstable")
plt.title("Power Producer by Class")
plt.xlabel("power_producer")
plt.legend()

# ── Chart 5: Reaction consumer1 by class ────────────────
plt.subplot(2, 3, 5)
sns.histplot(data[data["grid_status"]=="stable"]["reaction_consumer1"],
             bins=40, color="steelblue", alpha=0.6, label="Stable")
sns.histplot(data[data["grid_status"]=="unstable"]["reaction_consumer1"],
             bins=40, color="tomato", alpha=0.6, label="Unstable")
plt.title("Reaction Consumer1 by Class")
plt.xlabel("reaction_consumer1")
plt.legend()

# ── Chart 6: Correlation heatmap ─────────────────────────
plt.subplot(2, 3, 6)
cols_for_corr = ["reaction_producer", "power_producer",
                 "price_sensitivity_producer", "stability_score"]
corr = data[cols_for_corr].corr().round(2)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5)
plt.title("Correlation Heatmap")

plt.tight_layout()
plt.savefig("outputs/01_eda.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: outputs/01_eda.png")